PROJET : Système de Gestion de Coworking (SQL & Python)
TISNES Francesca - M1 APE - DS2E

Objectif : Démonstration d'une architecture SQL robuste incluant la gestion des espaces (Salles) et des flux financiers.

## 1. Initialisation du Schéma Relationnel

Pour répondre aux besoins de scalabilité, nous utilisons 4 tables :
1. **entreprises** : Les clients.
2. **membres** : Les salariés des entreprises.
3. **salles** : Le catalogue des espaces (Open Space, Bureaux, Réunions).
4. **reservations** : La table centrale liant les membres aux salles.

In [1]:
import sqlite3
import pandas as pd
from faker import Faker

# Connexion à la base
connection = sqlite3.connect("coworking_base.db")
cursor = connection.cursor()

# Nettoyage des anciennes tables pour repartir sur une base propre
cursor.execute("DROP TABLE IF EXISTS reservations")
cursor.execute("DROP TABLE IF EXISTS salles")
cursor.execute("DROP TABLE IF EXISTS membres")
cursor.execute("DROP TABLE IF EXISTS entreprises")

# Création des tables avec contraintes d'intégrité
cursor.execute("""
CREATE TABLE entreprises (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom TEXT NOT NULL,
    secteur TEXT,
    ville TEXT
)""")

cursor.execute("""
CREATE TABLE membres (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom_complet TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    entreprise_id INTEGER,
    FOREIGN KEY (entreprise_id) REFERENCES entreprises(id)
)""")

cursor.execute("""
CREATE TABLE salles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nom_salle TEXT NOT NULL,
    type_bureau TEXT CHECK(type_bureau IN ('Open Space', 'Bureau Privé', 'Salle de Réunion')),
    capacite INTEGER,
    tarif_heure REAL
)""")

cursor.execute("""
CREATE TABLE reservations (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    membre_id INTEGER,
    salle_id INTEGER,
    date_resa DATE,
    duree_heures INTEGER,
    FOREIGN KEY (membre_id) REFERENCES membres(id),
    FOREIGN KEY (salle_id) REFERENCES salles(id)
)""")

connection.commit()
print("Schéma SQL à 4 tables initialisé avec succès.")

Schéma SQL à 4 tables initialisé avec succès.


## 2. Peuplement de la Base (Données de test)
Nous utilisons la bibliothèque **Faker** pour générer un jeu de données réaliste : 10 entreprises, 40 membres et 100 réservations réparties sur nos 3 types de salles.

In [2]:
generator = Faker('fr_FR')

# 1. Insertion des Salles fixes
salles_data = [
    ('L\'Atelier', 'Open Space', 20, 5.0),
    ('Le Studio Secret', 'Bureau Privé', 2, 18.0),
    ('Grande Conférence', 'Salle de Réunion', 15, 45.0)
]
cursor.executemany("INSERT INTO salles (nom_salle, type_bureau, capacite, tarif_heure) VALUES (?,?,?,?)", salles_data)

# 2. Génération des Entreprises
for _ in range(10):
    cursor.execute("INSERT INTO entreprises (nom, secteur, ville) VALUES (?, ?, ?)", 
                   (generator.company(), generator.bs(), generator.city()))

# 3. Génération des Membres
for _ in range(40):
    cursor.execute("INSERT INTO membres (nom_complet, email, entreprise_id) VALUES (?, ?, ?)", 
                   (generator.name(), generator.email(), generator.random_int(min=1, max=10)))

# 4. Génération des Réservations
for _ in range(100):
    cursor.execute("INSERT INTO reservations (membre_id, salle_id, date_resa, duree_heures) VALUES (?, ?, ?, ?)", 
                   (generator.random_int(min=1, max=40), generator.random_int(min=1, max=3), 
                    generator.date_this_year(), generator.random_int(min=1, max=8)))

connection.commit()
print("Base de données alimentée (100 réservations).")

Base de données alimentée (100 réservations).


/tmp/ipykernel_1375/801374172.py:23: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute("INSERT INTO reservations (membre_id, salle_id, date_resa, duree_heures) VALUES (?, ?, ?, ?)",


## 3. Outils pour le Chef d'Entreprise
Cette section permet au gérant d'ajouter manuellement une entreprise ou une réservation sans passer par le script de simulation.

In [3]:
def chef_ajouter_entreprise(nom, secteur, ville):
    cursor.execute("INSERT INTO entreprises (nom, secteur, ville) VALUES (?, ?, ?)", (nom, secteur, ville))
    connection.commit()
    print(f"Succès : L'entreprise '{nom}' a été enregistrée manuellement.")

# Simulation d'un ajout manuel par le patron
chef_ajouter_entreprise("TISNES Coworking Services", "Logistique", "Strasbourg")

Succès : L'entreprise 'TISNES Coworking Services' a été enregistrée manuellement.


## 4. Analyse de l'Activité (Reporting)
Extraction du Chiffre d'Affaires cumulé par client via une **Triple Jointure SQL** et exportation automatique pour la comptabilité.

In [4]:
query = """
SELECT e.nom AS Client, SUM(s.tarif_heure * r.duree_heures) AS CA_Total_HT
FROM entreprises e
JOIN membres m ON e.id = m.entreprise_id
JOIN reservations r ON m.id = r.membre_id
JOIN salles s ON s.id = r.salle_id
GROUP BY e.nom
ORDER BY CA_Total_HT DESC
"""

df_final = pd.read_sql_query(query, connection)
df_final.to_csv("rapport_activite.csv", index=False)

# Affichage du top 5 des clients
df_final.head(5)

,Client,CA_Total_HT
0,Da Silva et Fils,1704.0
1,David SARL,1203.0
2,Devaux SA,1195.0
3,Paris SARL,1178.0
4,Reynaud,1016.0
